In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_percentage_error
from keras.models import Sequential
from keras.layers import SimpleRNN,Dense,Dropout
from keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

In [7]:
df=pd.read_csv("rnn_sequence_dataset.csv")

In [8]:
df

,t,value
0,0.000000,0.034092
1,0.125916,0.272815
2,0.251831,0.120151
3,0.377747,0.326518
4,0.503662,0.401622
...,...,...
495,62.328191,-0.483172
496,62.454106,-0.292410
497,62.580022,-0.225605
498,62.705938,-0.227046


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   t       500 non-null    float64
 1   value   500 non-null    float64
dtypes: float64(2)
memory usage: 7.9 KB


In [14]:
scaler=MinMaxScaler()
df["value"]=scaler.fit_transform(df[["value"]])

In [15]:
def create_dataset(data, window=20):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i+window])
        y.append(data[i+window])
    return np.array(X), np.array(y)



X,y=create_dataset(df["value"])


In [16]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,shuffle=False)


In [18]:
model=Sequential([
    SimpleRNN(64,input_shape=(20,1),return_sequences=True),
    SimpleRNN(32),
    Dense(1)
])

In [20]:
model.compile(optimizer="adam",loss="mse")

In [21]:
history=model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test,),callbacks=[EarlyStopping(monitor="val_loss",patience=3)])

Epoch 1/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - loss: 0.1285 - val_loss: 0.0388
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0213 - val_loss: 0.0145
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0078 - val_loss: 0.0053
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0065 - val_loss: 0.0092
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0059 - val_loss: 0.0030
Epoch 6/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0041 - val_loss: 0.0028
Epoch 7/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0041 - val_loss: 0.0022
Epoch 8/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0042 - val_loss: 0.0052
Epoch 9/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0040 - val_loss: 0.0023
Epoch 10/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0047 - val_loss: 0.0048


In [23]:
y_perd=model.predict(X_test)
y_pred=scaler.inverse_transform(y_perd)
y_test=scaler.inverse_transform(y_test.reshape(-1,1))

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


In [24]:
mean_absolute_percentage_error(y_test,y_pred)

0.7948456869269465

In [25]:
r2_score(y_test,y_pred)


0.9502287197087087